# CineStyle — Full Pipeline Notebook

> **AIPI 540 · Module Project · Duke University**

This notebook runs the complete CineStyle pipeline on a Colab A100:

| Step | What it does |
|------|--------------|
| 0 | Install dependencies |
| 1 | Clone repo & set up paths |
| 2 | Download Fashionpedia + build catalog |
| 3 | Enrich prices with Duke LiteLLM agent |
| 4 | Embed catalog with FashionCLIP → build FAISS index |
| 5 | Train NCF re-ranker (NeuMF, BPR loss) |
| 5b | Train SASRec re-ranker (Transformer, BCE loss) |
| 6 | Offline evaluation (Precision@K, Recall@K, NDCG@K, MAP@K) |
| 7 | Frame quality degradation experiment |
| 8 | Quick inference demo |

**Architecture overview:**
- **Stage 1** — FAISS KNN: item-item content-based retrieval (classical ML)
- **Stage 2** — NeuMF: GMF + MLP model-based CF (feedforward deep learning)
- **Stage 2b** — SASRec: causal self-attention over user sequences (Transformer)
- **Stage 3** — Diversity filter: price-spread deduplication

**Runtime:** ~60–75 min end-to-end on A100 with 20 k catalog items.

---


## 0 · Install dependencies

In [ ]:
%%capture
!pip install -q \
    torch torchvision \
    transformers \
    Pillow \
    faiss-gpu \
    datasets \
    huggingface-hub \
    tqdm \
    numpy \
    scikit-learn \
    scipy \
    requests \
    openai \
    pandas \
    matplotlib \
    python-dotenv

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## 1 · Clone repo & configure paths

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/Shreya-Mendi/CineStyle.git"
REPO_DIR = Path("/content/CineStyle")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print("Working dir:", Path.cwd())

In [ ]:
RAW_DIR   = Path("data/raw")
CROPS_DIR = RAW_DIR / "crops"
PROC_DIR  = Path("data/processed")
OUT_DIR   = Path("data/outputs")
IDX_DIR   = Path("models/faiss_index")
NCF_DIR   = Path("models/ncf_reranker")

for d in [RAW_DIR, CROPS_DIR, PROC_DIR, OUT_DIR, IDX_DIR, NCF_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Directories ready.")

## 2 · Download Fashionpedia & build catalog

Streams the Fashionpedia dataset, crops each annotated garment/accessory bounding box,
and writes `data/processed/catalog.jsonl`.

**Adjust `MAX_ITEMS`** — 5 000 is fast for testing; 20 000 is recommended for real retrieval quality.

In [ ]:
MAX_ITEMS            = 20_000   # garment crops to extract
N_USERS              = 500      # synthetic users for NCF training
INTERACTIONS_PER_USER = 30

In [ ]:
import json
import random
from datasets import load_dataset
from PIL import Image
from tqdm.notebook import tqdm

CATEGORY_NAMES = [
    "shirt, blouse", "top, t-shirt, sweatshirt", "sweater", "cardigan",
    "jacket", "vest", "pants", "shorts", "skirt", "coat", "dress", "jumpsuit",
    "cape", "glasses", "hat", "headband, head covering, hair accessory",
    "tie", "glove", "watch", "belt", "leg warmer", "tights, stockings",
    "sock", "shoe", "bag, wallet", "scarf", "umbrella", "hood", "collar",
    "lapel", "epaulette", "sleeve", "pocket", "neckline", "buckle", "zipper",
    "applique", "bead", "bow", "flower", "fringe", "ribbon", "rivet",
    "ruffle", "sequin", "tassel",
]

KEEP_CATEGORIES = {
    "shirt, blouse", "top, t-shirt, sweatshirt", "sweater", "cardigan",
    "jacket", "vest", "pants", "shorts", "skirt", "coat", "dress", "jumpsuit",
    "cape", "glasses", "hat", "headband, head covering, hair accessory",
    "tie", "glove", "watch", "belt", "tights, stockings", "sock", "shoe",
    "bag, wallet", "scarf",
}

AESTHETIC_MAP = {
    "dress": "feminine", "skirt": "feminine", "jumpsuit": "chic",
    "coat": "outerwear", "jacket": "outerwear", "cape": "outerwear",
    "cardigan": "casual", "sweater": "casual",
    "top, t-shirt, sweatshirt": "casual", "shirt, blouse": "smart casual",
    "vest": "smart casual", "pants": "casual", "shorts": "casual",
    "tights, stockings": "accessories", "sock": "accessories",
    "shoe": "accessories", "bag, wallet": "accessories",
    "watch": "accessories", "glasses": "accessories",
    "hat": "accessories",
    "headband, head covering, hair accessory": "accessories",
    "tie": "formal", "glove": "accessories",
    "belt": "accessories", "scarf": "accessories",
}

MIN_CROP_PX = 48

def category_name(idx):
    return CATEGORY_NAMES[idx] if 0 <= idx < len(CATEGORY_NAMES) else "unknown"


catalog_path = PROC_DIR / "catalog.jsonl"
count = 0

with open(catalog_path, "w") as f:
    for split in ("train", "val"):
        ds = load_dataset("detection-datasets/fashionpedia", split=split, streaming=True)
        for sample in tqdm(ds, desc=f"fashionpedia/{split}"):
            if count >= MAX_ITEMS:
                break
            pil_img = sample["image"]
            W, H = pil_img.width, pil_img.height
            objects = sample["objects"]
            for bbox_id, cat_idx, bbox in zip(
                objects["bbox_id"], objects["category"], objects["bbox"]
            ):
                if count >= MAX_ITEMS:
                    break
                cat = category_name(cat_idx)
                if cat not in KEEP_CATEGORIES:
                    continue
                x0, y0, x1, y1 = bbox
                cw, ch = x1 - x0, y1 - y0
                if cw < MIN_CROP_PX or ch < MIN_CROP_PX:
                    continue
                x0, y0 = max(0, int(x0)), max(0, int(y0))
                x1, y1 = min(W, int(x1)), min(H, int(y1))
                crop = pil_img.crop((x0, y0, x1, y1)).convert("RGB")
                crop_path = CROPS_DIR / f"{bbox_id}.jpg"
                crop.save(crop_path, "JPEG", quality=92)
                record = {
                    "id": str(bbox_id),
                    "title": cat.title(),
                    "brand": "",
                    "price": 0.0,
                    "image_url": str(crop_path),
                    "product_url": "",
                    "category": cat,
                    "aesthetic": AESTHETIC_MAP.get(cat, "fashion"),
                }
                f.write(json.dumps(record) + "\n")
                count += 1
        if count >= MAX_ITEMS:
            break

print(f"\nSaved {count} catalog items → {catalog_path}")

In [ ]:
# Assign mock prices now (will be overwritten by LLM agent in Step 3)
PRICE_RANGES = {
    "shoe": (60, 600), "bag, wallet": (50, 800), "watch": (80, 850),
    "coat": (80, 700), "jacket": (60, 600), "dress": (40, 500),
    "jumpsuit": (40, 400), "glasses": (20, 400), "hat": (20, 300),
    "belt": (15, 250), "scarf": (15, 200), "sweater": (30, 350),
    "cardigan": (30, 300), "shirt, blouse": (25, 300), "pants": (30, 300),
    "skirt": (25, 280), "top, t-shirt, sweatshirt": (15, 150),
    "shorts": (20, 150), "tights, stockings": (10, 80), "sock": (8, 40),
    "glove": (15, 150), "tie": (20, 200), "vest": (25, 200), "cape": (60, 500),
    "headband, head covering, hair accessory": (10, 100),
}

tmp = catalog_path.with_suffix(".jsonl.tmp")
with open(catalog_path) as fin, open(tmp, "w") as fout:
    for line in fin:
        item = json.loads(line)
        lo, hi = PRICE_RANGES.get(item["category"], (20, 300))
        item["price"] = round(random.uniform(lo, hi), 2)
        fout.write(json.dumps(item) + "\n")
tmp.replace(catalog_path)
print("Mock prices assigned.")

In [ ]:
# Build synthetic interactions for NCF training
with open(catalog_path) as f:
    catalog = [json.loads(l) for l in f]

by_cat = {}
for item in catalog:
    by_cat.setdefault(item["category"], []).append(item["id"])

all_ids   = [item["id"] for item in catalog]
categories = list(by_cat.keys())

interactions_path = PROC_DIR / "interactions.jsonl"
with open(interactions_path, "w") as f:
    for user_id in tqdm(range(N_USERS), desc="interactions"):
        fav_cats = random.sample(categories, k=min(random.randint(1, 3), len(categories)))
        fav_pool = [iid for c in fav_cats for iid in by_cat[c]]
        n_fav  = int(INTERACTIONS_PER_USER * 0.7)
        n_rand = INTERACTIONS_PER_USER - n_fav
        selected = set(random.sample(fav_pool, k=min(n_fav, len(fav_pool))))
        selected |= set(random.sample(all_ids,  k=min(n_rand, len(all_ids))))
        for item_id in selected:
            f.write(json.dumps({"user_id": user_id, "item_id": item_id, "label": 1}) + "\n")

print(f"Interactions saved → {interactions_path}")

## 3 · Enrich prices with Duke LiteLLM agent

The agent calls `https://litellm.oit.duke.edu/v1` with tool-calling.
Set your API key below. If you skip this cell, mock prices from Step 2 are used.

In [ ]:
import os

DUKE_LLM_API_KEY = ""          # ← paste your Duke LiteLLM API key here
DUKE_LLM_MODEL   = "claude-opus-4-6"   # any model on litellm.oit.duke.edu
USE_WEB_SEARCH   = False        # set True to let agent search Google for grounding

os.environ["DUKE_LLM_API_KEY"] = DUKE_LLM_API_KEY
os.environ["DUKE_LLM_MODEL"]   = DUKE_LLM_MODEL

In [ ]:
import re
import time
import requests as _requests
from openai import OpenAI

DUKE_BASE_URL = "https://litellm.oit.duke.edu/v1"

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "web_search_prices",
            "description": "Search Google for retail prices of a fashion item.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "emit_price_range",
            "description": "Emit the final price range for the category.",
            "parameters": {
                "type": "object",
                "properties": {
                    "category":   {"type": "string"},
                    "price_low":  {"type": "number"},
                    "price_high": {"type": "number"},
                    "reasoning":  {"type": "string"},
                },
                "required": ["category", "price_low", "price_high", "reasoning"],
            },
        },
    },
]

SYSTEM_PROMPT = (
    "You are a fashion retail pricing expert. Determine a realistic USD retail price range "
    "(budget to mid-luxury) for a garment/accessory category.\n\n"
    "Rules:\n"
    "- price_low  = ~10th percentile typical retail (Zara/H&M tier)\n"
    "- price_high = ~90th percentile (Nordstrom tier, not couture)\n"
    "- Optionally call web_search_prices once to ground your estimate, then emit_price_range.\n"
    "- If confident, go straight to emit_price_range."
)

FALLBACK = {
    "dress": (40, 500), "shirt, blouse": (25, 300),
    "top, t-shirt, sweatshirt": (15, 150), "sweater": (30, 350),
    "cardigan": (30, 300), "jacket": (60, 600), "coat": (80, 700),
    "vest": (25, 200), "pants": (30, 300), "shorts": (20, 150),
    "skirt": (25, 280), "jumpsuit": (40, 400), "cape": (60, 500),
    "shoe": (60, 600), "bag, wallet": (50, 800), "watch": (80, 850),
    "belt": (15, 250), "scarf": (15, 200), "glasses": (20, 400),
    "hat": (20, 300), "headband, head covering, hair accessory": (10, 100),
    "glove": (15, 150), "tie": (20, 200), "tights, stockings": (10, 80),
    "sock": (8, 40),
}


def _google_text(query):
    headers = {"User-Agent": "Mozilla/5.0", "Accept-Language": "en-US,en;q=0.9"}
    try:
        r = _requests.get("https://www.google.com/search",
                          headers=headers, params={"q": query, "num": 8}, timeout=10)
        text = re.sub(r"<[^>]+>", " ", r.text)
        return re.sub(r"\s+", " ", text)[:2000]
    except Exception as e:
        return f"[error: {e}]"


def _run_agent(client, category, model, use_web):
    tools = TOOLS if use_web else [TOOLS[1]]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Price range for: **{category}** (women's fashion, US market)."},
    ]
    for _ in range(5):
        resp = client.chat.completions.create(
            model=model, messages=messages, tools=tools,
            tool_choice="auto", max_tokens=512, temperature=0.2,
        )
        msg = resp.choices[0].message
        messages.append(msg)
        if resp.choices[0].finish_reason in ("stop", None) or not msg.tool_calls:
            break
        price_range = None
        results = []
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            if tc.function.name == "emit_price_range":
                price_range = {"price_low": float(args["price_low"]),
                               "price_high": float(args["price_high"]),
                               "reasoning": args.get("reasoning", "")}
                content = "Recorded."
            elif tc.function.name == "web_search_prices" and use_web:
                content = _google_text(args["query"])
            else:
                content = "[unavailable]"
            results.append({"role": "tool", "tool_call_id": tc.id, "content": content})
        messages.extend(results)
        if price_range:
            return price_range
    return None


def fetch_and_enrich(catalog_path, model, use_web, delay=1.0):
    api_key = os.environ.get("DUKE_LLM_API_KEY", "")
    if not api_key:
        print("No DUKE_LLM_API_KEY set — keeping mock prices.")
        return

    with open(catalog_path) as f:
        all_cats = list({json.loads(l)["category"] for l in f})

    client = OpenAI(api_key=api_key, base_url=DUKE_BASE_URL)
    ranges = {}

    print(f"Running price agent for {len(all_cats)} categories (model={model}) ...")
    for cat in tqdm(all_cats, desc="price agent"):
        try:
            result = _run_agent(client, cat, model=model, use_web=use_web)
            if result:
                lo, hi = max(5.0, result["price_low"]), min(2000.0, result["price_high"])
                ranges[cat] = (lo, hi)
                print(f"  {cat}: ${lo:.0f}–${hi:.0f} | {result['reasoning'][:70]}")
            else:
                ranges[cat] = FALLBACK.get(cat, (20, 300))
        except Exception as e:
            ranges[cat] = FALLBACK.get(cat, (20, 300))
            print(f"  {cat}: error ({e}) — fallback")
        time.sleep(delay)

    tmp = catalog_path.with_suffix(".jsonl.tmp")
    with open(catalog_path) as fin, open(tmp, "w") as fout:
        for line in fin:
            item = json.loads(line)
            lo, hi = ranges.get(item["category"], (20, 300))
            item["price"] = round(random.uniform(lo, hi), 2)
            fout.write(json.dumps(item) + "\n")
    tmp.replace(catalog_path)
    print(f"\nPrices updated → {catalog_path}")


fetch_and_enrich(catalog_path, model=DUKE_LLM_MODEL, use_web=USE_WEB_SEARCH)

## 4 · Embed catalog with FashionCLIP → build FAISS index

Uses `patrickjohncyh/fashion-clip` (CLIP fine-tuned on 700 K Farfetch items).
Batched inference with `torch.cuda.amp.autocast` for A100 throughput.
Produces a 512-dim FAISS `IndexFlatIP` (cosine similarity).

In [ ]:
import torch
import numpy as np
import faiss
from transformers import CLIPModel, CLIPProcessor
from PIL import Image

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "patrickjohncyh/fashion-clip"
BATCH_SIZE = 128   # tune down if OOM
DUMMY_TEXT = ["fashion item"]

print(f"Loading {MODEL_NAME} on {DEVICE} ...")
clip_model     = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained(MODEL_NAME)
clip_model.eval()
print("Model loaded.")

In [ ]:
def get_image_embeds(images, model, processor, device):
    """Return L2-normalised image embeddings. Shape: (N, 512)."""
    inputs = processor(
        images=images,
        text=DUMMY_TEXT * len(images),
        return_tensors="pt",
        padding=True,
    ).to(device)
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=(device == "cuda")):
        out = model(**inputs)
    feats = out.image_embeds          # (N, 512)
    return feats / feats.norm(dim=-1, keepdim=True)


def get_text_embeds(labels, model, processor, device):
    """Return L2-normalised text embeddings. Shape: (N, 512)."""
    dummy_img = Image.new("RGB", (224, 224))
    inputs = processor(
        images=[dummy_img] * len(labels),
        text=labels,
        return_tensors="pt",
        padding=True,
    ).to(device)
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=(device == "cuda")):
        out = model(**inputs)
    feats = out.text_embeds
    return feats / feats.norm(dim=-1, keepdim=True)


print("Embedding functions ready.")

In [ ]:
with open(catalog_path) as f:
    catalog = [json.loads(l) for l in f]

DIM   = 512
index = faiss.IndexFlatIP(DIM)
meta  = []

img_batch  = []
meta_batch = []

def flush_batch():
    feats = get_image_embeds(img_batch, clip_model, clip_processor, DEVICE)
    arr   = feats.cpu().float().numpy().astype(np.float32)
    faiss.normalize_L2(arr)
    index.add(arr)
    meta.extend(meta_batch)
    img_batch.clear()
    meta_batch.clear()

skipped = 0
for item in tqdm(catalog, desc="Embedding catalog"):
    try:
        img = Image.open(item["image_url"]).convert("RGB")
    except Exception:
        skipped += 1
        continue
    img_batch.append(img)
    meta_batch.append({
        "id": item["id"], "title": item["title"],
        "brand": item.get("brand", ""), "price": item["price"],
        "image_url": item["image_url"], "product_url": item.get("product_url", ""),
        "category": item.get("category", ""),
    })
    if len(img_batch) >= BATCH_SIZE:
        flush_batch()

if img_batch:
    flush_batch()

faiss.write_index(index, str(IDX_DIR / "products.index"))
with open(IDX_DIR / "meta.json", "w") as f:
    json.dump(meta, f)

print(f"\nIndex built: {index.ntotal} vectors  |  skipped: {skipped}")
print(f"Saved → {IDX_DIR}/products.index")

## 5 · Train NCF re-ranker

**NeuMF** (He et al. 2017): GMF branch + MLP branch combined.
Trained with Bayesian Personalized Ranking (BPR) loss on synthetic interactions.

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# ── Model ──────────────────────────────────────────────────────────────────
class NeuMF(nn.Module):
    """
    Neural Matrix Factorization.
    Input : (user_id [long], item_embed [float32, 512-dim])
    Output: score scalar per (user, item) pair
    """
    def __init__(self, n_users, item_dim=512, embed_dim=64,
                 mlp_dims=None):
        super().__init__()
        mlp_dims = mlp_dims or [256, 128, 64]

        self.user_gmf  = nn.Embedding(n_users, embed_dim)
        self.item_gmf  = nn.Linear(item_dim, embed_dim, bias=False)
        self.user_mlp  = nn.Embedding(n_users, embed_dim)
        self.item_mlp  = nn.Linear(item_dim, embed_dim, bias=False)

        layers, in_dim = [], embed_dim * 2
        for out_dim in mlp_dims:
            layers += [nn.Linear(in_dim, out_dim), nn.ReLU()]
            in_dim = out_dim
        self.mlp    = nn.Sequential(*layers)
        self.output = nn.Linear(embed_dim + mlp_dims[-1], 1)
        nn.init.xavier_uniform_(self.output.weight)

    def forward(self, user_ids, item_embeds):
        gmf_out = self.user_gmf(user_ids) * self.item_gmf(item_embeds)
        mlp_out = self.mlp(torch.cat([self.user_mlp(user_ids),
                                       self.item_mlp(item_embeds)], dim=-1))
        return self.output(torch.cat([gmf_out, mlp_out], dim=-1)).squeeze(-1)


def bpr_loss(pos, neg):
    return -torch.log(torch.sigmoid(pos - neg) + 1e-8).mean()


# ── Dataset ────────────────────────────────────────────────────────────────
class BPRDataset(Dataset):
    def __init__(self, interactions_path, embeddings):
        with open(interactions_path) as f:
            self.rows = [json.loads(l) for l in f]
        self.embeddings = embeddings
        self.item_ids   = list(embeddings.keys())

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row    = self.rows[idx]
        pos_id = row["item_id"]
        neg_id = self.item_ids[np.random.randint(len(self.item_ids))]
        pos_emb = self.embeddings.get(pos_id, np.zeros(512, dtype=np.float32))
        neg_emb = self.embeddings.get(neg_id, np.zeros(512, dtype=np.float32))
        return (
            torch.tensor(row["user_id"], dtype=torch.long),
            torch.tensor(pos_emb, dtype=torch.float32),
            torch.tensor(neg_emb, dtype=torch.float32),
        )

print("NCF classes defined.")

In [ ]:
NCF_EPOCHS     = 10
NCF_BATCH_SIZE = 256
NCF_LR         = 1e-3

# Load stored embeddings from FAISS index
stored_index = faiss.read_index(str(IDX_DIR / "products.index"))
with open(IDX_DIR / "meta.json") as f:
    meta_list = json.load(f)

xb_ptr = stored_index.get_xb()
xb     = faiss.rev_swig_ptr(xb_ptr, stored_index.ntotal * DIM).reshape(stored_index.ntotal, DIM)
embeddings = {meta_list[i]["id"]: xb[i].copy() for i in range(len(meta_list))}

with open(interactions_path) as f:
    interactions = [json.loads(l) for l in f]
n_users = max(r["user_id"] for r in interactions) + 1

ncf_model  = NeuMF(n_users=n_users).to(DEVICE)
optimizer  = torch.optim.Adam(ncf_model.parameters(), lr=NCF_LR)
dataset    = BPRDataset(interactions_path, embeddings)
loader     = DataLoader(dataset, batch_size=NCF_BATCH_SIZE, shuffle=True, num_workers=2)

loss_history = []
print(f"Training NeuMF  |  n_users={n_users}  |  interactions={len(interactions)}")

for epoch in range(1, NCF_EPOCHS + 1):
    ncf_model.train()
    total = 0.0
    for user_ids, pos_embs, neg_embs in loader:
        user_ids = user_ids.to(DEVICE)
        pos_embs = pos_embs.to(DEVICE)
        neg_embs = neg_embs.to(DEVICE)
        pos_scores = ncf_model(user_ids, pos_embs)
        neg_scores = ncf_model(user_ids, neg_embs)
        loss = bpr_loss(pos_scores, neg_scores)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
    avg = total / len(loader)
    loss_history.append(avg)
    print(f"  Epoch {epoch:02d}/{NCF_EPOCHS}  loss={avg:.4f}")

torch.save(ncf_model.state_dict(), NCF_DIR / "ncf.pt")
torch.save({"n_users": n_users}, NCF_DIR / "config.pt")
print(f"\nSaved NCF weights → {NCF_DIR}/ncf.pt")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 3))
plt.plot(range(1, len(loss_history) + 1), loss_history, marker="o", color="#d97706")
plt.xlabel("Epoch")
plt.ylabel("BPR Loss")
plt.title("NCF Re-ranker Training Loss")
plt.tight_layout()
plt.savefig(OUT_DIR / "ncf_loss.png", dpi=120)
plt.show()

## 5b · Train SASRec re-ranker (Transformer)

**SASRec** (Kang & McAuley, ICDM 2018): Self-Attentive Sequential Recommendation.

Architecture:
- FashionCLIP embeddings (512-dim) → linear projection → d_model=128
- Learned positional embeddings
- N=2 causal Transformer blocks (multi-head self-attention + FFN + LayerNorm)
- Trained with binary cross-entropy on (sequence → next item) prediction
- At inference: sequence repr (last position) · target item proj → score

This is the **Transformer / Sequential** tier from the course slides (slide 35).


In [ ]:
import math

# ── SASRec Transformer block ────────────────────────────────────────────────
class SASRecBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ln1  = nn.LayerNorm(d_model)
        self.ffn  = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )
        self.ln2  = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, causal_mask):
        h = self.ln1(x)
        h, _ = self.attn(h, h, h, attn_mask=causal_mask, is_causal=False)
        x = x + self.drop(h)
        h = self.ln2(x)
        x = x + self.ffn(h)
        return x


class SASRec(nn.Module):
    """
    Self-Attentive Sequential Recommendation.
    Input : sequence of FashionCLIP embeddings (B, L, 512)
    Output: relevance score (B,) against a target item embedding
    """
    def __init__(self, item_dim=512, d_model=128, n_heads=4,
                 n_layers=2, max_seq_len=50, dropout=0.2):
        super().__init__()
        self.d_model     = d_model
        self.max_seq_len = max_seq_len
        self.item_proj   = nn.Linear(item_dim, d_model, bias=False)
        self.pos_emb     = nn.Embedding(max_seq_len, d_model)
        self.emb_drop    = nn.Dropout(dropout)
        self.blocks      = nn.ModuleList(
            [SASRecBlock(d_model, n_heads, dropout) for _ in range(n_layers)]
        )
        self.ln_out      = nn.LayerNorm(d_model)
        self.target_proj = nn.Linear(item_dim, d_model, bias=False)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def _causal_mask(self, L, device):
        return torch.triu(torch.ones(L, L, device=device), diagonal=1).bool()

    def encode_sequence(self, seq_embeds):
        B, L, _ = seq_embeds.shape
        x = self.item_proj(seq_embeds)
        pos_ids = torch.arange(L, device=seq_embeds.device).unsqueeze(0)
        x = self.emb_drop(x + self.pos_emb(pos_ids))
        mask = self._causal_mask(L, seq_embeds.device)
        for block in self.blocks:
            x = block(x, mask)
        return self.ln_out(x)

    def forward(self, seq_embeds, target_embeds):
        """seq_embeds: (B,L,512)  target_embeds: (B,512)  → scores (B,)"""
        last   = self.encode_sequence(seq_embeds)[:, -1, :]   # (B, d_model)
        target = self.target_proj(target_embeds)               # (B, d_model)
        return (last * target).sum(dim=-1)                     # (B,)


# ── SASRec dataset ──────────────────────────────────────────────────────────
class SASRecDataset(Dataset):
    """(sequence → next-item) pairs from interaction file."""
    def __init__(self, interactions_path, embeddings, max_seq_len=50):
        self.embeddings  = embeddings
        self.max_seq_len = max_seq_len
        self.item_ids    = list(embeddings.keys())

        user_items = {}
        with open(interactions_path) as f:
            for line in f:
                row = json.loads(line)
                user_items.setdefault(row["user_id"], []).append(row["item_id"])

        self.samples = []
        for item_list in user_items.values():
            for i in range(1, len(item_list)):
                seq = item_list[max(0, i - max_seq_len): i]
                pos = item_list[i]
                if pos in embeddings:
                    self.samples.append((seq, pos))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        seq_ids, pos_id = self.samples[idx]
        neg_id = self.item_ids[np.random.randint(len(self.item_ids))]
        L = self.max_seq_len
        seq_embs = np.zeros((L, 512), dtype=np.float32)
        for j, iid in enumerate(seq_ids[-L:]):
            emb = self.embeddings.get(iid)
            if emb is not None:
                seq_embs[L - len(seq_ids) + j] = emb
        pos_emb = self.embeddings.get(pos_id, np.zeros(512, dtype=np.float32))
        neg_emb = self.embeddings.get(neg_id, np.zeros(512, dtype=np.float32))
        return (
            torch.tensor(seq_embs, dtype=torch.float32),
            torch.tensor(pos_emb,  dtype=torch.float32),
            torch.tensor(neg_emb,  dtype=torch.float32),
        )

print("SASRec classes defined.")


In [ ]:
SASREC_EPOCHS   = 20
SASREC_BATCH    = 256
SASREC_LR       = 1e-3
SASREC_D_MODEL  = 128
SASREC_N_HEADS  = 4
SASREC_N_LAYERS = 2
SASREC_SEQ_LEN  = 50
SASREC_DROPOUT  = 0.2

SASREC_DIR = Path("models/sasrec")
SASREC_DIR.mkdir(parents=True, exist_ok=True)

sasrec_model = SASRec(
    d_model=SASREC_D_MODEL,
    n_heads=SASREC_N_HEADS,
    n_layers=SASREC_N_LAYERS,
    max_seq_len=SASREC_SEQ_LEN,
    dropout=SASREC_DROPOUT,
).to(DEVICE)

sasrec_dataset = SASRecDataset(interactions_path, embeddings, max_seq_len=SASREC_SEQ_LEN)
sasrec_loader  = DataLoader(sasrec_dataset, batch_size=SASREC_BATCH,
                             shuffle=True, num_workers=2, pin_memory=True)

sasrec_optimizer = torch.optim.Adam(sasrec_model.parameters(),
                                    lr=SASREC_LR, betas=(0.9, 0.98))
sasrec_scheduler = torch.optim.lr_scheduler.OneCycleLR(
    sasrec_optimizer, max_lr=SASREC_LR,
    epochs=SASREC_EPOCHS, steps_per_epoch=max(1, len(sasrec_dataset) // SASREC_BATCH),
)
bce_loss = nn.BCEWithLogitsLoss()

sasrec_loss_history = []
n_params = sum(p.numel() for p in sasrec_model.parameters() if p.requires_grad)
print(f"SASRec | samples={len(sasrec_dataset)} | params={n_params:,} | device={DEVICE}")

for epoch in range(1, SASREC_EPOCHS + 1):
    sasrec_model.train()
    total = 0.0
    for seq_embs, pos_embs, neg_embs in sasrec_loader:
        seq_embs = seq_embs.to(DEVICE)
        pos_embs = pos_embs.to(DEVICE)
        neg_embs = neg_embs.to(DEVICE)

        pos_scores = sasrec_model(seq_embs, pos_embs)
        neg_scores = sasrec_model(seq_embs, neg_embs)
        loss = (bce_loss(pos_scores, torch.ones_like(pos_scores)) +
                bce_loss(neg_scores, torch.zeros_like(neg_scores)))

        sasrec_optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(sasrec_model.parameters(), 1.0)
        sasrec_optimizer.step()
        sasrec_scheduler.step()
        total += loss.item()

    avg = total / len(sasrec_loader)
    sasrec_loss_history.append(avg)
    print(f"  Epoch {epoch:02d}/{SASREC_EPOCHS}  loss={avg:.4f}")

torch.save(sasrec_model.state_dict(), SASREC_DIR / "sasrec.pt")
torch.save({
    "d_model": SASREC_D_MODEL, "n_heads": SASREC_N_HEADS,
    "n_layers": SASREC_N_LAYERS, "max_seq_len": SASREC_SEQ_LEN,
    "dropout": SASREC_DROPOUT,
}, SASREC_DIR / "config.pt")
print(f"\nSaved SASRec weights → {SASREC_DIR}/sasrec.pt")

# Plot training loss
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(range(1, len(loss_history)+1), loss_history, marker="o", color="#d97706")
axes[0].set_title("NeuMF BPR Loss"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[1].plot(range(1, len(sasrec_loss_history)+1), sasrec_loss_history, marker="s", color="#2563eb")
axes[1].set_title("SASRec BCE Loss"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
plt.tight_layout()
plt.savefig(OUT_DIR / "training_loss.png", dpi=120)
plt.show()


## 6 · Offline Evaluation

Metrics: Precision@K, Recall@K, NDCG@K, MAP@K (K = 5, 10).

Uses last 20% of each user's interactions as held-out ground truth.
Query embedding = the last seen training item's embedding.

In [ ]:
import math

def precision_at_k(recs, relevant, k):
    return sum(1 for r in recs[:k] if r["id"] in relevant) / k

def recall_at_k(recs, relevant, k):
    if not relevant: return 0.0
    return sum(1 for r in recs[:k] if r["id"] in relevant) / len(relevant)

def dcg_at_k(recs, relevant, k):
    return sum(1.0 / math.log2(i + 2)
               for i, r in enumerate(recs[:k]) if r["id"] in relevant)

def ndcg_at_k(recs, relevant, k):
    ideal = sum(1.0 / math.log2(i + 2)
                for i in range(min(len(relevant), k)))
    return dcg_at_k(recs, relevant, k) / ideal if ideal > 0 else 0.0

def avg_precision(recs, relevant):
    hits, ap = 0, 0.0
    for i, r in enumerate(recs):
        if r["id"] in relevant:
            hits += 1
            ap += hits / (i + 1)
    return ap / len(relevant) if relevant else 0.0

print("Metric helpers ready.")

In [ ]:
def faiss_retrieve(embedding, top_k=50):
    q = embedding.reshape(1, -1).astype(np.float32)
    faiss.normalize_L2(q)
    D, I = stored_index.search(q, top_k)
    results = []
    for dist, idx in zip(D[0], I[0]):
        if idx < 0: continue
        item = dict(meta_list[idx])
        item["similarity"] = float(dist)
        results.append(item)
    return results

In [ ]:
# ── SASRec evaluation helper ──────────────────────────────────────────────
def sasrec_rerank(sasrec_model, candidates, user_history_embs, device, max_seq_len=50):
    """Re-rank candidates with SASRec given user interaction history."""
    if not user_history_embs or not candidates:
        return candidates
    L = max_seq_len
    history = user_history_embs[-L:]
    seq_np = np.zeros((1, L, 512), dtype=np.float32)
    for j, h in enumerate(history):
        seq_np[0, L - len(history) + j] = h
    seq_tensor = torch.tensor(seq_np, dtype=torch.float32).to(device)
    seq_batch  = seq_tensor.expand(len(candidates), -1, -1)
    target_embs = torch.tensor(
        np.array([embeddings.get(c["id"], np.zeros(512, dtype=np.float32))
                  for c in candidates]),
        dtype=torch.float32,
    ).to(device)
    sasrec_model.eval()
    with torch.no_grad():
        scores = sasrec_model(seq_batch, target_embs).cpu().numpy()
    for c, s in zip(candidates, scores):
        c["sasrec_score"] = float(s)
    return sorted(candidates, key=lambda x: x.get("sasrec_score", 0), reverse=True)


# ── Evaluate all three stages ──────────────────────────────────────────────
K_VALUES = [5, 10]
N_EVAL   = 200

results_faiss   = {k: {"precision": [], "recall": [], "ndcg": []} for k in K_VALUES}
results_sasrec  = {k: {"precision": [], "recall": [], "ndcg": []} for k in K_VALUES}
all_recs_f, all_recs_s, all_rel = [], [], []

for user_id, item_ids in tqdm(list(user_items.items())[:N_EVAL], desc="Evaluating"):
    split      = max(1, int(len(item_ids) * 0.8))
    train_ids  = item_ids[:split]
    test_ids   = set(item_ids[split:])
    if not test_ids or len(train_ids) < 2:
        continue
    q_idx = id_to_idx.get(train_ids[-1])
    if q_idx is None:
        continue
    query_emb    = xb[q_idx]
    history_embs = [xb[id_to_idx[i]] for i in train_ids[:-1] if i in id_to_idx]

    recs_f = faiss_retrieve(query_emb, top_k=max(K_VALUES))
    recs_s = sasrec_rerank(sasrec_model, list(recs_f), history_embs,
                            DEVICE, SASREC_SEQ_LEN)

    all_recs_f.append(recs_f); all_recs_s.append(recs_s); all_rel.append(test_ids)

    for k in K_VALUES:
        results_faiss[k]["precision"].append(precision_at_k(recs_f, test_ids, k))
        results_faiss[k]["recall"].append(recall_at_k(recs_f, test_ids, k))
        results_faiss[k]["ndcg"].append(ndcg_at_k(recs_f, test_ids, k))
        results_sasrec[k]["precision"].append(precision_at_k(recs_s, test_ids, k))
        results_sasrec[k]["recall"].append(recall_at_k(recs_s, test_ids, k))
        results_sasrec[k]["ndcg"].append(ndcg_at_k(recs_s, test_ids, k))

map_faiss  = np.mean([avg_precision(r, rel) for r, rel in zip(all_recs_f, all_rel)])
map_sasrec = np.mean([avg_precision(r, rel) for r, rel in zip(all_recs_s, all_rel)])

summary = {}
print("\n{'Model':<12} {'Metric':<15} Value")
print("-" * 40)
for label, results_k in [("FAISS KNN", results_faiss), ("SASRec", results_sasrec)]:
    for k in K_VALUES:
        for metric in ["precision", "recall", "ndcg"]:
            key = f"{label} {metric.capitalize()}@{k}"
            val = float(np.mean(results_k[k][metric]))
            summary[key] = val
            print(f"{label:<12} {metric.capitalize()+'@'+str(k):<15} {val:.4f}")

summary["FAISS MAP@10"]  = float(map_faiss)
summary["SASRec MAP@10"] = float(map_sasrec)
print(f"{'FAISS KNN':<12} {'MAP@10':<15} {map_faiss:.4f}")
print(f"{'SASRec':<12} {'MAP@10':<15} {map_sasrec:.4f}")

with open(OUT_DIR / "eval_results.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nSaved → {OUT_DIR}/eval_results.json")


In [ ]:
labels = list(summary.keys())
values = list(summary.values())

fig, ax = plt.subplots(figsize=(9, 3))
bars = ax.barh(labels, values, color="#d97706", alpha=0.85)
ax.set_xlim(0, max(values) * 1.3)
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", fontsize=9)
ax.set_xlabel("Score")
ax.set_title("CineStyle — Offline Retrieval Metrics")
plt.tight_layout()
plt.savefig(OUT_DIR / "eval_chart.png", dpi=120)
plt.show()

## 7 · Frame quality degradation experiment

Tests how HD / JPEG-compressed / Gaussian-blurred inputs affect retrieval precision.
Uses the first 20 catalog images as test probes (no external test images needed).

In [ ]:
from PIL import ImageFilter
import io as _io

def degrade(img, mode):
    if mode == "hd":
        return img
    if mode == "compressed":
        buf = _io.BytesIO()
        img.save(buf, format="JPEG", quality=15)
        buf.seek(0)
        return Image.open(buf).convert("RGB")
    if mode == "blurred":
        return img.filter(ImageFilter.GaussianBlur(radius=4))
    raise ValueError(mode)


MODES      = ["hd", "compressed", "blurred"]
EVAL_K     = 10
TEST_ITEMS = [m for m in meta_list if Path(m["image_url"]).exists()][:20]

deg_results = {m: [] for m in MODES}

for item in tqdm(TEST_ITEMS, desc="Degradation experiment"):
    img = Image.open(item["image_url"]).convert("RGB")
    embs = {}
    for mode in MODES:
        deg_img = degrade(img, mode)
        feat = get_image_embeds([deg_img], clip_model, clip_processor, DEVICE)
        embs[mode] = feat.squeeze(0).cpu().float().numpy()

    hd_emb  = embs["hd"]
    hd_ids  = {r["id"] for r in faiss_retrieve(hd_emb, top_k=EVAL_K)}

    for mode in MODES:
        e    = embs[mode]
        cos  = float(np.dot(hd_emb, e) / (np.linalg.norm(hd_emb) * np.linalg.norm(e) + 1e-8))
        recs = faiss_retrieve(e, top_k=EVAL_K)
        p    = precision_at_k(recs, hd_ids, EVAL_K)
        deg_results[mode].append({"cosine_vs_hd": cos, f"P@{EVAL_K}_vs_hd": p})

deg_summary = {}
print("\n=== Frame Quality Degradation ===")
for mode in MODES:
    cos_mean = np.mean([r["cosine_vs_hd"] for r in deg_results[mode]])
    p_mean   = np.mean([r[f"P@{EVAL_K}_vs_hd"] for r in deg_results[mode]])
    deg_summary[mode] = {"mean_cosine_vs_hd": float(cos_mean),
                         f"mean_P@{EVAL_K}_vs_hd": float(p_mean)}
    print(f"  {mode:<12}  cosine={cos_mean:.4f}  P@{EVAL_K}={p_mean:.4f}")

with open(OUT_DIR / "experiment_frame_quality.json", "w") as f:
    json.dump(deg_summary, f, indent=2)
print(f"\nSaved → {OUT_DIR}/experiment_frame_quality.json")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

cos_vals = [deg_summary[m]["mean_cosine_vs_hd"] for m in MODES]
p_vals   = [deg_summary[m][f"mean_P@{EVAL_K}_vs_hd"] for m in MODES]

axes[0].bar(MODES, cos_vals, color="#d97706", alpha=0.85)
axes[0].set_title("Mean cosine similarity vs HD")
axes[0].set_ylim(0, 1.05)

axes[1].bar(MODES, p_vals, color="#92400e", alpha=0.85)
axes[1].set_title(f"Mean P@{EVAL_K} vs HD retrieval")
axes[1].set_ylim(0, 1.05)

for ax in axes:
    ax.set_xlabel("Frame quality")

plt.suptitle("Frame Quality Degradation Experiment", fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / "experiment_chart.png", dpi=120)
plt.show()

## 8 · Quick inference demo

Pick any catalog image, embed it, and retrieve the top-10 most similar items.

In [ ]:
GARMENT_LABELS = [
    "dress", "top", "blouse", "jacket", "coat", "pants", "jeans",
    "skirt", "shorts", "suit", "sweater", "hoodie", "shirt",
    "bag", "shoes", "hat", "scarf", "belt", "watch", "glasses",
]
COLOR_LABELS = [
    "black", "white", "red", "blue", "green", "yellow",
    "pink", "purple", "brown", "grey", "beige", "navy",
]
AESTHETIC_LABELS = [
    "casual", "formal", "streetwear", "bohemian", "minimalist",
    "old money", "dark academia", "cottagecore", "y2k", "preppy",
]


def embed_and_classify(img):
    img_feat = get_image_embeds([img], clip_model, clip_processor, DEVICE).squeeze(0)

    def classify(labels):
        txt = get_text_embeds(labels, clip_model, clip_processor, DEVICE)
        sims = (img_feat.unsqueeze(0) @ txt.T).squeeze(0)
        return labels[sims.argmax().item()]

    return {
        "garment_type": classify(GARMENT_LABELS),
        "color":        classify(COLOR_LABELS),
        "aesthetic":    classify(AESTHETIC_LABELS),
        "embedding":    img_feat.cpu().float().numpy(),
    }


# Pick a dress item as the demo probe
probe_meta = next((m for m in meta_list if "dress" in m.get("category", "").lower()), meta_list[0])
probe_img  = Image.open(probe_meta["image_url"]).convert("RGB")

result = embed_and_classify(probe_img)
print("Garment type :", result["garment_type"])
print("Color        :", result["color"])
print("Aesthetic    :", result["aesthetic"])

In [ ]:
TOP_K = 10
recs  = faiss_retrieve(result["embedding"], top_k=TOP_K)

print(f"\nTop {TOP_K} recommendations for probe image:")
print(f"{'Rank':<5} {'Category':<30} {'Price':>8} {'Similarity':>12}")
print("-" * 60)
for i, r in enumerate(recs, 1):
    print(f"{i:<5} {r.get('category', r['title']):<30} "
          f"${r['price']:>7.2f} {r['similarity']:>12.4f}")

In [ ]:
import math

valid = [r for r in recs if Path(r["image_url"]).exists()][:8]
n     = len(valid)
cols  = 4
rows  = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.5))
axes = axes.flatten() if n > 1 else [axes]

for ax, rec in zip(axes, valid):
    img = Image.open(rec["image_url"]).convert("RGB")
    ax.imshow(img)
    ax.set_title(f"{rec.get('category', rec['title']).title()}\n"
                 f"${rec['price']:.0f}  sim={rec['similarity']:.3f}",
                 fontsize=8)
    ax.axis("off")

for ax in axes[len(valid):]:
    ax.axis("off")

plt.suptitle("CineStyle — Top Recommendations", fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / "demo_recommendations.png", dpi=120)
plt.show()

---

## Pipeline complete

Artifacts saved to `data/outputs/`:

| File | Contents |
|------|----------|
| `training_loss.png` | NeuMF BPR + SASRec BCE training curves |
| `eval_results.json` | FAISS vs SASRec Precision@K, Recall@K, NDCG@K, MAP@K |
| `eval_chart.png` | Retrieval metrics bar chart |
| `experiment_frame_quality.json` | Frame degradation experiment results |
| `experiment_chart.png` | Degradation charts |
| `demo_recommendations.png` | Visual recommendation grid |

Models saved to `models/`:

| File | Contents |
|------|----------|
| `models/faiss_index/products.index` | FAISS product embedding index |
| `models/faiss_index/meta.json` | Product metadata |
| `models/ncf_reranker/ncf.pt` | NeuMF weights |
| `models/ncf_reranker/config.pt` | NCF config (n_users) |
| `models/sasrec/sasrec.pt` | SASRec Transformer weights |
| `models/sasrec/config.pt` | SASRec config (d_model, n_heads, n_layers, seq_len) |

**Three model tiers (rubric compliance):**
- Naive baseline: popularity scoring
- Classical ML: FAISS KNN over FashionCLIP embeddings
- Deep learning (feedforward): NeuMF (GMF + MLP, BPR loss)
- Deep learning (Transformer): SASRec (causal self-attention, BCE loss)

To serve the FastAPI backend:
```bash
uvicorn main:app --reload --port 8000
```
